# Parda — do the context rules cut false bars without hiding less real PII?

Runs both benchmarks with the context rules (`parda/pii/context.py`: dates of birth need a birth label, unspaced
12-digit Aadhaar needs an Aadhaar label, no spans cutting through numbers) and compares with the stored results.
**Keep the rules only if** "all PII fully hidden" drops by at most 0.2 points and over-redaction goes down.

**Settings:** Accelerator **None (CPU)** · Internet **On**
**Inputs:** Your Work → **Parda** (Phase 1) · **08b** · **10** · Datasets tab → `celeba-dataset` · **Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`
Run cells 1–3 by hand, then **Save Version → Save & Run All** (~1.5 h).


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts (to regenerate the unseen pages) + GLiNER, YOLO, ONNX; the shipped models from the PUBLIC repo
!bash scripts/setup_kaggle.sh
!pip install -q gliner ultralytics onnxruntime
import json, os, shutil
from huggingface_hub import snapshot_download
W = "/kaggle/working"
snapshot_download("chetan-0804/parda-web-models", local_dir=f"{W}/models",
                  allow_patterns=["gliner/model_int8_embed.onnx", "gliner/parda_onnx.json", "yolo/parda-yolo.onnx"])
meta = json.load(open(f"{W}/models/gliner/parda_onnx.json")); meta["default"] = "int8_embed"
meta["files"] = {k: v for k, v in meta["files"].items() if k in ("int8_embed",)} | {"int8_embed": "model_int8_embed.onnx"}
json.dump(meta, open(f"{W}/models/gliner/parda_onnx.json", "w"))
!python tests/test_context.py


In [ ]:
# 3. Inputs: data, faces, saved OCR (08b), stored results without the rules (08b: full Parda, 10: browser build)
from parda.fsutil import find_path
from parda.ocr.run_ocr import find_data, _n_images
from parda.vision.faces import find_faces_dir
IN = "/kaggle/input"
first = lambda *rels: next((h[0] for r in rels for h in [find_path(IN, r)] if h), None)
DATA = find_data(); FACES = find_faces_dir(IN, exclude=[DATA])
C_SEEN = os.path.dirname(first("cache_seen/ocr_easyocr.jsonl") or "")
C_OOD = os.path.dirname(first("cache_ood/ocr_easyocr.jsonl") or "")
BASE = {"seen": first("compress_results/report_seen.json"), "unseen": first("compress_results/report_unseen.json")}
for k, v in [("DATA", DATA), ("FACES", FACES), ("OCR seen", C_SEEN), ("OCR unseen", C_OOD), ("results seen", BASE["seen"]), ("results unseen", BASE["unseen"])]:
    print(f"{k:16s} {v}")
assert DATA and _n_images(DATA) == 30000 and FACES and C_SEEN and C_OOD and BASE["seen"] and BASE["unseen"]


In [ ]:
# 4. Both benchmarks WITH the context rules (shipped ONNX models, saved OCR) (~70 min)
!python -m parda.vision.yolo_data --data {DATA} --out {W}/yolo_bench --faces_dir {FACES} --only bench --workers 4 > /dev/null
!python -m parda.synth.ood --n 300 --out {W}/ood --faces_dir {FACES} --workers 4 > /dev/null
for b, src in (("seen", C_SEEN), ("unseen", C_OOD)):
    os.makedirs(f"{W}/cache_{b}", exist_ok=True)
    for f in ("ocr_tesseract.jsonl", "ocr_easyocr.jsonl"):
        shutil.copy(f"{src}/{f}", f"{W}/cache_{b}/{f}")
!python -m parda.pipeline.evaluate_e2e --data {DATA} --yolo_data {W}/yolo_bench --ocr tesseract,easyocr --gliner_ft {W}/models/gliner --yolo {W}/models/yolo/parda-yolo.onnx --cache {W}/cache_seen --context dob,aadhaar,cut --out {W}/ctx_seen --title "Context rules — seen templates" 2>&1 | grep -v -i warning | tail -6
!python -m parda.pipeline.evaluate_e2e --data {W}/ood --ocr tesseract,easyocr --gliner_ft {W}/models/gliner --yolo {W}/models/yolo/parda-yolo.onnx --cache {W}/cache_unseen --context dob,aadhaar,cut --out {W}/ctx_unseen --title "Context rules — unseen layouts + phone photos" 2>&1 | grep -v -i warning | tail -6


In [ ]:
# 5. Verdict: without -> with the context rules
from IPython.display import Markdown, display
pct = lambda v: f"{100 * v:.1f}%"
rows = ["| Benchmark | Configuration | All PII fully hidden | Over-redaction | Pages with no leak |", "|---|---|---|---|---|"]
ok = True
for b in ("seen", "unseen"):
    old = {r["config"]: r for r in json.load(open(BASE[b]))["configs"]}
    new = {r["config"]: r for r in json.load(open(f"{W}/ctx_{b}/report_e2e.json"))["configs"]}
    for name in ("+ YOLO visual = Parda", "browser build: Tesseract only + YOLO"):
        o, n = old[name], new[name]
        d_pii, d_over = n["fully_redacted"] - o["fully_redacted"], n["over_redaction"] - o["over_redaction"]
        ok &= d_pii >= -0.002 and d_over <= 0
        rows.append(f"| {b} | {name} | {pct(o['fully_redacted'])} → **{pct(n['fully_redacted'])}** ({100 * d_pii:+.1f}) | "
                    f"{pct(o['over_redaction'])} → **{pct(n['over_redaction'])}** ({100 * d_over:+.1f}) | {pct(o['pages_clean'])} → {pct(n['pages_clean'])} |")
verdict = ("**Keep the context rules**: PII hidden stays within 0.2 points and over-redaction goes down." if ok else
           "**Do not turn them on by default**: they hide less real PII or do not reduce over-redaction. (Per-rule runs: --context dob / aadhaar / cut.)")
report = "\n".join(["### Context rules (dates of birth, unspaced Aadhaar, spans cutting numbers)", "", *rows, "", verdict]) + "\n"
open(f"{W}/context_rules.md", "w").write(report)
display(Markdown(report))
R = f"{W}/context_results"; os.makedirs(R, exist_ok=True)
shutil.copy(f"{W}/context_rules.md", R)
for b in ("seen", "unseen"):
    for ext in ("md", "json"):
        shutil.copy(f"{W}/ctx_{b}/report_e2e.{ext}", f"{R}/report_{b}.{ext}")
print(sorted(os.listdir(R)))
